[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C36_GPU_Kernels_Course/03_tiled_matmul/03_tiled_matmul.ipynb)

# 03 · 分块矩阵乘（用 numpy 模拟 shared-memory tiling）

目标：亲手写出 **分块矩阵乘**，与 `A @ B` 对拍到 `1e-10`，并用**算术强度 / roofline** 量化「分块为什么快」。

路线：朴素 GEMM 参考 → 算术强度 → 分块 GEMM（核心，对拍 A@B）→ 复用计数 → roofline → register tiling 复用 → ✏️ 练习 → 📖 答案 → 🧪 真实模型 GEMM 胶囊。

> 心智模型：**一个 block 算 C 的一个小块；a_tile/b_tile = 搬进 shared 的子矩阵；`__syncthreads()` = 我们先整块切片再整块计算的顺序**。我们写的是分块*结构*，不是性能。

## 1 · 朴素 GEMM：我们的 ground-truth 参考

最朴素的三重循环：每个 `C[i,j]` 读 A 的整行、B 的整列，做 K 次乘加。它慢，但**绝对正确**，是后面所有「优化版」对拍的基准。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def matmul_naive(A, B):
    '''三重循环 GEMM。每个输出元素读 A 一行 + B 一列（= 朴素 GPU 内核里一个线程干的活）。'''
    M, K = A.shape
    K2, N = B.shape
    assert K == K2, 'A 的列数必须等于 B 的行数'
    C = np.zeros((M, N))
    for i in range(M):
        for j in range(N):
            acc = 0.0
            for k in range(K):
                acc += A[i, k] * B[k, j]
            C[i, j] = acc
    return C

A = rng.standard_normal((6, 5))
B = rng.standard_normal((5, 4))
C = matmul_naive(A, B)
print('形状', C.shape, '| 对拍 A@B :', np.allclose(C, A @ B, atol=1e-12))
assert np.allclose(C, A @ B, atol=1e-12)
print('✅ 朴素 GEMM 与 numpy A@B 一致 —— 它将作为 ground truth')

## 2 · 算术强度：朴素 GEMM 为何访存受限

FLOPs = 2·M·N·K（每输出元素 K 乘 + K 加）是**定数**。朴素写法对 A 的每个元素读 N 次、对 B 读 M 次，HBM 字节 ≈ `4·2MNK`（FP32）。

算术强度 = FLOPs / bytes，会得到一个**与矩阵大小无关、恒约 0.25** 的小数 —— 铁定落在脊点左侧。

In [ ]:
def gemm_flops(M, N, K):
    return 2 * M * N * K            # 每输出元素 K 次乘 + K 次加

def ai_naive(M, N, K, dtype_bytes=4):
    flops = gemm_flops(M, N, K)
    # 朴素：A 读 N 次(每列重读整个 A)，B 读 M 次 -> 共 2MNK 个元素从 HBM 进来
    bytes_moved = dtype_bytes * (2 * M * N * K)
    return flops / bytes_moved

for (M, N, K) in [(128,128,128), (512,512,512), (4096,4096,4096)]:
    print(f'M=N=K={K:5d} -> 算术强度 {ai_naive(M,N,K):.3f} FLOP/byte')
assert abs(ai_naive(512, 512, 512) - 0.25) < 1e-9   # 2MNK / (4·2MNK) = 1/4
print('\n关键：无论矩阵多大，朴素 GEMM 算术强度恒 = 0.25 FLOP/byte。')
print('对比 A100 脊点 ~150 FLOP/byte —— 朴素 GEMM 性能上限 < 峰值的 0.2%！')

## 3 · 分块 GEMM：把数据搬进 shared、复用、再换

**本模块的核心。** 把 C 切成 `TILE×TILE` 的小块（一个 block 算一块）；沿 K 方向逐段把 `a_tile`、`b_tile`「搬进 shared」（numpy 里 = 切片），让小块内所有元素复用它们，累加。

`acc += a_tile @ b_tile` 这一步，就是 block 内 TILE² 个线程各自对这一段做点积的**向量化写法**。结果必须逐位等于 `A@B`。

In [ ]:
def matmul_tiled(A, B, TILE=16):
    '''模拟 shared-memory 分块 GEMM。
       外两层循环 = 遍历 C 的输出小块(每块一个 block)；
       内层 k0 循环 = 沿 K 逐段搬 tile 进 shared 并累加(== __syncthreads 循环)。'''
    M, K = A.shape
    K2, N = B.shape
    assert K == K2
    C = np.zeros((M, N))
    for i0 in range(0, M, TILE):                 # 遍历输出小块的行
        for j0 in range(0, N, TILE):             # 遍历输出小块的列
            mb = min(TILE, M - i0)               # 处理边界：最后一块可能不满
            nb = min(TILE, N - j0)
            acc = np.zeros((mb, nb))             # block 的累加器(寄存器)
            for k0 in range(0, K, TILE):         # 沿 K 逐段
                a_tile = A[i0:i0+TILE, k0:k0+TILE]  # 搬进 shared 的 A 子块
                b_tile = B[k0:k0+TILE, j0:j0+TILE]  # 搬进 shared 的 B 子块
                acc += a_tile @ b_tile           # block 复用这两个 tile
            C[i0:i0+TILE, j0:j0+TILE] = acc      # 扫完 K，落一次 global
    return C

# 故意用非 TILE 整数倍的形状，逼出边界处理
A = rng.standard_normal((40, 48))
B = rng.standard_normal((48, 33))
for TILE in [8, 16, 32]:
    C = matmul_tiled(A, B, TILE=TILE)
    ok = np.allclose(C, A @ B, atol=1e-10)
    print(f'TILE={TILE:2d} -> 对拍 A@B : {ok}')
    assert ok, f'分块结果必须等于 A@B (TILE={TILE})'
print('✅ 分块 GEMM 在多种 TILE、非整除形状下都逐位等于 A@B')

> 关键点：`acc` 在整个 K-循环里累加，**只在最后写一次 `C`**（落一次 global）；`a_tile`/`b_tile` 每段搬一次、被 `acc` 的所有元素复用。这正是 shared-memory tiling 省带宽的来源。边界靠 `min(...)` 与切片自动处理。

## 4 · 复用计数：分块到底省了多少 HBM 访问

分块把 A 的每个元素读取次数从 N 降到 `N/TILE`、B 从 M 降到 `M/TILE`。HBM 字节随之缩小约 TILE 倍，算术强度放大约 TILE 倍。

In [ ]:
def ai_tiled(M, N, K, TILE, dtype_bytes=4):
    flops = gemm_flops(M, N, K)
    # A 被读 N/TILE 次 -> M*K*(N/TILE) 个元素；B 被读 M/TILE 次 -> K*N*(M/TILE)；C 写一次 M*N
    elems = M * K * (N // TILE) + K * N * (M // TILE) + M * N
    return flops / (dtype_bytes * elems)

M = N = K = 1024
print(f'朴素           : {ai_naive(M,N,K):6.2f} FLOP/byte')
for TILE in [8, 16, 32, 64]:
    print(f'分块 TILE={TILE:3d}   : {ai_tiled(M,N,K,TILE):6.2f} FLOP/byte  '
          f'(↑ {ai_tiled(M,N,K,TILE)/ai_naive(M,N,K):.0f}x vs 朴素)')
assert ai_tiled(M, N, K, 32) > ai_naive(M, N, K)
assert ai_tiled(M, N, K, 64) > ai_tiled(M, N, K, 16)   # TILE 越大 AI 越高
print('\n✅ 算术强度随 TILE 单调上升 —— 这就是分块把内核推过脊点的旋钮')

## 5 · Roofline：把算术强度翻译成「能达到多少 FLOP/s」

可达性能 = `min(峰值算力, 算术强度 × 带宽)`。脊点 `AI* = 峰值算力 / 带宽`：AI 在脊点左 → 访存受限（被斜坡压着）；在脊点右 → 算力受限（顶到天花板）。

用 A100 的真实参数（FP16 峰值 312 TFLOP/s、HBM 带宽 2.039 TB/s）算给朴素 vs 分块。

In [ ]:
def ridge_point(peak_flops, bandwidth):
    return peak_flops / bandwidth                 # 脊点算术强度 (FLOP/byte)

def attainable_flops(ai, peak_flops, bandwidth):
    return min(peak_flops, ai * bandwidth)        # roofline 折线

A100_PEAK = 312e12        # FP16 TFLOP/s
A100_BW   = 2.039e12      # HBM 带宽 byte/s
ridge = ridge_point(A100_PEAK, A100_BW)
print(f'A100 脊点算术强度 AI* = {ridge:.0f} FLOP/byte\n')

M = N = K = 4096
for label, ai in [('朴素', ai_naive(M,N,K)),
                  ('分块 TILE=32', ai_tiled(M,N,K,32)),
                  ('分块 TILE=128', ai_tiled(M,N,K,128))]:
    perf = attainable_flops(ai, A100_PEAK, A100_BW)
    bound = '算力受限' if ai >= ridge else '访存受限'
    print(f'{label:14s} AI={ai:7.2f} -> 可达 {perf/1e12:7.1f} TFLOP/s  ({perf/A100_PEAK:5.1%} 峰值)  [{bound}]')

assert ai_naive(M,N,K) < ridge, '朴素应在脊点左侧(访存受限)'
assert attainable_flops(ai_naive(M,N,K), A100_PEAK, A100_BW) < 0.01 * A100_PEAK
assert attainable_flops(ai_tiled(M,N,K,128), A100_PEAK, A100_BW) > 50 * attainable_flops(ai_naive(M,N,K), A100_PEAK, A100_BW)
print('\n✅ 朴素 GEMM 上限不到峰值 1%；单层分块把可达性能抬高几十倍(但仍未顶到天花板——需 FP16+多级 tiling)')

## 6 · register tiling：在 shared 之上再榨一层复用

光靠 shared 还不够顶。让**一个线程算一个 `C×C` 的输出微块**（而非单个元素）：每段内层循环读入 C 个 A 值、C 个 B 值，却做 `C×C` 次 FMA —— 操作数从寄存器里被复用了 C 次。这就是真实 GEMM 的 thread/register tile 这一级。

In [ ]:
def operand_reuse(micro):
    '''一个线程算 micro×micro 输出微块时，每个从 shared 读入寄存器的操作数被复用几次。
       内层一步：读 micro 个 A + micro 个 B (共 2*micro 次读)，做 micro*micro 次 FMA。
       每个操作数复用次数 = micro。'''
    reads = 2 * micro
    fmas = micro * micro
    return fmas / reads            # = micro/2 * ... 实际每操作数复用 micro 次

for micro in [1, 2, 4, 8]:
    fma_per_read = operand_reuse(micro)
    print(f'微块 {micro}x{micro}: 每次寄存器读支撑 {fma_per_read:.1f} 次 FMA (操作数复用 {micro}x)')
assert operand_reuse(4) > operand_reuse(1)
assert operand_reuse(8) > operand_reuse(4)        # 微块越大复用越多
print('\n✅ register tiling 让复用在 shared 复用之上再翻几倍 —— 代价是寄存器需求暴涨(回咬占用率)')

---
## ✏️ 练习 1：从零实现分块 GEMM

不看上面的实现，自己把 `matmul_tiled` 写出来。要点：外两层遍历输出小块、内层沿 K 逐段累加、用 `min(...)` 处理边界、`acc` 只在最后写回 `C`。

必须对多种 TILE 与**非整除**形状都等于 `A@B`。

In [ ]:
def matmul_tiled_ex(A, B, TILE=16):
    M, K = A.shape
    K2, N = B.shape
    assert K == K2
    C = np.zeros((M, N))
    # TODO: 三层分块循环：
    #   for i0 in range(0, M, TILE):
    #     for j0 in range(0, N, TILE):
    #        acc = zeros(min(TILE,M-i0), min(TILE,N-j0))
    #        for k0 in range(0, K, TILE):
    #            acc += A[i0:i0+TILE, k0:k0+TILE] @ B[k0:k0+TILE, j0:j0+TILE]
    #        C[i0:i0+TILE, j0:j0+TILE] = acc
    raise NotImplementedError
    return C

In [ ]:
# —— 练习 1 自测 ——
A = rng.standard_normal((37, 41))     # 故意都不是 TILE 整数倍
B = rng.standard_normal((41, 29))
for TILE in [4, 8, 16, 32, 64]:
    C = matmul_tiled_ex(A, B, TILE=TILE)
    assert C.shape == (37, 29)
    assert np.allclose(C, A @ B, atol=1e-10), f'TILE={TILE} 应等于 A@B'
print('✅ 练习 1 通过：分块 GEMM 在所有 TILE 与非整除形状下都对拍 A@B')

## ✏️ 练习 2：求使内核「算力受限」的最小 TILE

给定 GPU 的峰值算力与带宽，分块 GEMM 要跨过脊点（`AI ≥ AI*`）才算力受限。

实现 `min_tile_compute_bound(M,N,K,peak,bw)`：返回使 `ai_tiled ≥ ridge_point` 的最小 TILE（从一组候选里选）。复用 `ai_tiled`、`ridge_point`。

In [ ]:
def min_tile_compute_bound(M, N, K, peak, bw, candidates=(8,16,32,64,128,256)):
    # TODO: 算 ridge = ridge_point(peak, bw)；遍历 candidates，
    #       返回第一个使 ai_tiled(M,N,K,TILE) >= ridge 的 TILE；都不行返回 None
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
M = N = K = 8192
# 一个脊点较低的设定，便于小 TILE 跨过
peak, bw = 100e12, 2e12          # 脊点 = 50 FLOP/byte
t = min_tile_compute_bound(M, N, K, peak, bw)
ridge = ridge_point(peak, bw)
assert t is not None
assert ai_tiled(M, N, K, t) >= ridge, '返回的 TILE 应使 AI 跨过脊点'
# 再小一档的候选应当还没跨过(确认是“最小”)
smaller = [x for x in (8,16,32,64,128,256) if x < t]
if smaller:
    assert ai_tiled(M, N, K, smaller[-1]) < ridge, '它应当是跨过脊点的最小 TILE'
print(f'脊点 AI*={ridge:.0f} FLOP/byte -> 最小算力受限 TILE = {t}')
print('✅ 练习 2 通过')

## ✏️ 练习 3：roofline 分类器

实现 `classify(ai, peak, bw)`：返回 `'compute'`（算力受限，`ai≥脊点`）或 `'memory'`（访存受限）。

然后对一组算子按 A100 分类。**注意一个反直觉的事实**：A100 的脊点高达 ~153 FLOP/byte，所以*单层* FP32 分块（即便 TILE 不小）算术强度只到几十，**仍然 memory-bound**！真正跨过脊点要靠 **FP16/BF16（字节减半，AI 翻倍）+ 多级 register tiling**（见下面那条 ~300 的多级内核）。这正是为什么 cuBLAS 要把分块叠那么多层。

In [ ]:
def classify(ai, peak, bw):
    # TODO: ai >= ridge_point(peak,bw) -> 'compute'，否则 'memory'
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
peak, bw = 312e12, 2.039e12       # A100，脊点 ≈ 153 FLOP/byte
kernels = {
    '逐元素 add':            0.25,
    '朴素 GEMM':             ai_naive(4096,4096,4096),
    '单层分块 FP32 TILE=64': ai_tiled(4096,4096,4096,64),       # ~16
    '单层分块 FP32 TILE=256':ai_tiled(4096,4096,4096,256),      # ~62
    '多级 FP16 (CUTLASS式)': 300.0,   # FP16 + 多级 register tiling 才能到这个量级
}
got = {name: classify(ai, peak, bw) for name, ai in kernels.items()}
for k, v in got.items():
    print(f'{k:24s} -> {v}')
assert got['逐元素 add'] == 'memory'
assert got['朴素 GEMM'] == 'memory'
assert got['单层分块 FP32 TILE=256'] == 'memory', '单层 FP32 分块到几十 FLOP/byte，仍在 A100 脊点(153)左侧'
assert got['多级 FP16 (CUTLASS式)'] == 'compute', '只有 FP16 + 多级 tiling 才跨过脊点'
print('\n✅ 练习 3 通过：单层分块还不够跨过现代 GPU 的高脊点 —— 这就是多级 tiling 与低精度存在的理由')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def matmul_tiled_ex(A, B, TILE=16):
    M, K = A.shape
    K2, N = B.shape
    assert K == K2
    C = np.zeros((M, N))
    for i0 in range(0, M, TILE):
        for j0 in range(0, N, TILE):
            mb = min(TILE, M - i0); nb = min(TILE, N - j0)
            acc = np.zeros((mb, nb))
            for k0 in range(0, K, TILE):
                acc += A[i0:i0+TILE, k0:k0+TILE] @ B[k0:k0+TILE, j0:j0+TILE]
            C[i0:i0+TILE, j0:j0+TILE] = acc
    return C

In [ ]:
# 练习 2 参考答案
def min_tile_compute_bound(M, N, K, peak, bw, candidates=(8,16,32,64,128,256)):
    ridge = ridge_point(peak, bw)
    for TILE in sorted(candidates):
        if ai_tiled(M, N, K, TILE) >= ridge:
            return TILE
    return None

In [ ]:
# 练习 3 参考答案
def classify(ai, peak, bw):
    return 'compute' if ai >= ridge_point(peak, bw) else 'memory'

---
## 🧪 真实数据胶囊：真实模型里的 GEMM 算一笔账

大模型每一层都是巨型 GEMM。用 **Llama-2-7B** 的真实维度（hidden=4096、FFN 中间维=11008）算几个典型 GEMM 的 FLOPs 与算术强度，对比单层 FP32 分块 vs FP16（字节减半），看清「为什么真实 GEMM 必须用 FP16 + 多级 tiling 才能跑满 H100」。

In [ ]:
# Llama-2-7B 真实超参 + 一批 token 的常见 GEMM 形状
HIDDEN = 4096
FFN    = 11008
TOKENS = 4096          # batch*seq 的 token 数(训练里很常见)

GEMMS = {
    'QKV 投影  (T,H)x(H,3H)': (TOKENS, 3 * HIDDEN, HIDDEN),
    'MLP up    (T,H)x(H,F)':  (TOKENS, FFN, HIDDEN),
    'MLP down  (T,F)x(F,H)':  (TOKENS, HIDDEN, FFN),
}

H100_PEAK, H100_BW = 989e12, 3.35e12
ridge = ridge_point(H100_PEAK, H100_BW)
print(f'H100 脊点 AI* = {ridge:.0f} FLOP/byte (非常高！)\n')
print(f"{'GEMM':26s} {'GFLOP':>8s} {'AI(FP32,T=128)':>15s} {'AI(FP16,T=128)':>15s}")
for name, (M, N, K) in GEMMS.items():
    fl = gemm_flops(M, N, K)
    ai32 = ai_tiled(M, N, K, TILE=128, dtype_bytes=4)
    ai16 = ai_tiled(M, N, K, TILE=128, dtype_bytes=2)   # FP16: 字节减半 -> AI 翻倍
    print(f'{name:26s} {fl/1e9:8.1f} {ai32:15.1f} {ai16:15.1f}')
print('\n观察：单层分块(FP32~32、FP16~63 FLOP/byte)都还在 H100 脊点(295)左侧 —— 仍访存受限！')
print('真实 cuBLAS 靠 FP16 + 多级 register tiling 把 AI 再抬几倍才跨过脊点。')
print('这正是 GEMM 内核要把分块叠到四五层、并死磕 Tensor Core 的原因。')

**🧪 胶囊练习**：实现 `mlp_block_flops(tokens, hidden, ffn)`：算 Llama 风格 MLP **一层**（up + gate + down 三个 GEMM；SwiGLU 有 up 和 gate 两个 `H→F`，再一个 `F→H`）的总 FLOPs。

In [ ]:
def mlp_block_flops(tokens, hidden, ffn):
    # SwiGLU MLP: gate=(T,H)@(H,F), up=(T,H)@(H,F), down=(T,F)@(F,H)
    # TODO: 用 gemm_flops 把这三个 GEMM 的 FLOPs 加起来
    raise NotImplementedError

In [ ]:
# 自测
f = mlp_block_flops(TOKENS, HIDDEN, FFN)
expected = gemm_flops(TOKENS, FFN, HIDDEN) * 2 + gemm_flops(TOKENS, HIDDEN, FFN)
assert f == expected
print(f'Llama-2-7B 一层 MLP({TOKENS} tokens) ≈ {f/1e9:.0f} GFLOP')
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def mlp_block_flops(tokens, hidden, ffn):
    gate = gemm_flops(tokens, ffn, hidden)
    up   = gemm_flops(tokens, ffn, hidden)
    down = gemm_flops(tokens, hidden, ffn)
    return gate + up + down

---
## 🔧 旁注：对应的 Triton GEMM 内核长什么样

本课 numpy 模拟的「一个 block 算一个 C 小块、沿 K 逐段累加」，在 Triton 里就是经典的 matmul 内核（伪代码，**本环境不跑**）：

```python
import triton, triton.language as tl

@triton.autotune(configs=[...], key=['M', 'N', 'K'])   # 自动搜 BLOCK_M/N/K
@triton.jit
def matmul_kernel(a_ptr, b_ptr, c_ptr, M, N, K,
                  BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr):
    pid_m = tl.program_id(0)      # 我负责 C 的哪一个行块  (== 我们的 i0/TILE)
    pid_n = tl.program_id(1)      # 哪一个列块            (== 我们的 j0/TILE)
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)   # == 我们的 acc
    for k0 in range(0, K, BLOCK_K):                        # == 我们的 K 逐段循环
        a = tl.load(a_ptr + ...)   # 搬 A 子块(带 mask 边界保护)
        b = tl.load(b_ptr + ...)   # 搬 B 子块
        acc += tl.dot(a, b)        # tl.dot 映射到 Tensor Core(== 我们的 a_tile@b_tile)
    tl.store(c_ptr + ..., acc, mask=...)                   # 落一次 global
```

对应关系：`program_id`↔输出小块下标、`for k0`↔K 逐段累加、`tl.dot`↔`a_tile @ b_tile`（且自动用 Tensor Core）、`acc`↔我们的累加器。Triton 把「搬进 shared、同步、bank 布局、双缓冲」自动处理掉——你只需写对**分块结构**，正是本课练的东西。`@triton.autotune` 替你搜最优 TILE。

### 小结
- 朴素 GEMM 算术强度恒 = 0.25 FLOP/byte，铁定**访存受限**(性能上限不到峰值 1%)，病根是数据零复用。
- **shared-memory tiling**：一个 block 算一个 C 小块，把 a_tile/b_tile 搬进 shared 复用 TILE 次 → 算术强度 O(1)→O(TILE)。
- **roofline**：可达性能 = min(峰值算力, AI×带宽)；脊点 AI*=峰值/带宽。TILE 是把 GEMM 从脊点左推到右的旋钮。
- 选 TILE = 复用(要大)对抗占用率(要小)；真实 GEMM 叠 **多级分块 + register tiling + 双缓冲 + Tensor Core**(cuBLAS/CUTLASS)。

下一站：**模块 04 · 并行规约与融合 softmax** —— 学完点积式的复用，接着学如何并行地求和/求最大，为 FlashAttention 备料。